In [1]:
import re
import pandas as pd
import argparse

def chunk_by_turns(df):
    """Splits text by conversational turns within each Thread ID."""
    all_chunks = []
    for _, row in df.iterrows():
        thread_id = row['Thread ID']
        text = row['Text']
        message_id = row.get('Message ID', None)  # Handles missing Message ID
        # Split by common conversational starts (expand as needed)
        turns = re.split(r'(?i)(?=\b(?:hi|hello|thanks|okay|good morning|good evening|pleasure|cheers)\b)', text)
        turns = [t.strip() for t in turns if t.strip()]  # Remove empty strings

        for turn in turns:
            all_chunks.append({'Thread ID': thread_id, 'Message ID': message_id, 'Text': turn})
    return pd.DataFrame(all_chunks)

def main():
    parser = argparse.ArgumentParser(description="Chunk conversation data by turns.")
    parser.add_argument("input_file", help="Path to the input CSV file.")
    parser.add_argument("output_file", help="Path to the output CSV file.")
    args = parser.parse_args()

    try:
        # Read the input CSV, handling potential encoding issues
        try:
            df = pd.read_csv(args.input_file)
        except UnicodeDecodeError:
            df = pd.read_csv(args.input_file, encoding='latin1')  # Try latin1 encoding
        except UnicodeDecodeError:
            df = pd.read_csv(args.input_file, encoding='utf-8-sig') #Try utf-8-sig to handle BOM
            

        if 'Thread ID' not in df.columns or 'Text' not in df.columns:
            raise ValueError("Input CSV must contain 'Thread ID' and 'Text' columns.")

        chunked_df = chunk_by_turns(df)
        chunked_df.to_csv(args.output_file, index=False)
        print(f"Chunked data saved to {args.output_file}")

    except FileNotFoundError:
        print(f"Error: Input file '{args.input_file}' not found.")
    except pd.errors.EmptyDataError:
        print(f"Error: Input file '{args.input_file}' is empty.")
    except ValueError as e:
        print(f"ValueError: {e}")
    except Exception as e: #Catching any other exceptions
        print(f"An unexpected error occurred: {e}")


if __name__ == "__main__":
    main()

usage: ipykernel_launcher.py [-h] input_file output_file
ipykernel_launcher.py: error: the following arguments are required: input_file, output_file


SystemExit: 2

C:\Users\jatro\AppData\Roaming\Python\Python313\site-packages\IPython\core\interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
